<a href="https://colab.research.google.com/github/Smyles019/html-login-form-detector/blob/feature/random-forest-model/notebooks/random_forest_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


In [ ]:
# Clone the repository if running in Colab
import sys
if 'google.colab' in sys.modules:
    !git clone -b feature/random-forest-model https://github.com/Smyles019/html-login-form-detector.git


# Random Forest Training Pipeline
This notebook orchestrates the training and evaluation of a Random Forest model on the HTML login form dataset.
It uses modular Python code to ensure reproducibility and prevent data leakage.


In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd()
if cwd.name == 'notebooks':
    PROJECT_ROOT = cwd.parent
elif (cwd / 'src').exists():
    PROJECT_ROOT = cwd
else:
    PROJECT_ROOT = cwd / "html-login-form-detector"

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

print(f"Project root set to: {PROJECT_ROOT}")


## 1. Load Data & Explore
We load the explicit `train` and `validation` datasets separately, just like the XGBoost pipeline, to ensure a valid 1:1 comparison.


In [ ]:
from preprocessing.html_preprocessing import load_data, preprocess_and_split
from random_forest.train_random_forest import train_random_forest
from random_forest.evaluate_random_forest import evaluate_random_forest

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "Dataset 1"
df_train, df_val = load_data(str(DATA_DIR))

print(f"Train records: {len(df_train)}")
print(f"Validation records: {len(df_val)}")


## 2. Preprocess & Scale
Here we fit the `CountVectorizer` and `RobustScaler` on the training dataset strictly, and apply it to the validation dataset to prevent data leakage.


In [ ]:
X_train, X_test, y_train, y_test, feature_names, label_encoder = preprocess_and_split(df_train, df_val)
print(f"Training dimensions: {X_train.shape}")
print(f"Testing dimensions: {X_test.shape}")
print(f"Number of features: {len(feature_names)}")


## 3. Train Random Forest Model


In [ ]:
MODEL_SAVE_PATH = PROJECT_ROOT / "models" / "random_forest_model.joblib"
model = train_random_forest(X_train, y_train, str(MODEL_SAVE_PATH))


## 4. Evaluation & Feature Importance


In [ ]:
RESULTS_DIR = PROJECT_ROOT / "results"
metrics, cm_df, fi_df = evaluate_random_forest(
    model, X_test, y_test, feature_names, label_encoder.classes_, str(RESULTS_DIR)
)
